# 05 — Date Utilities and SQL Interoperability

Demonstrates PySpark date/time handling and the SQL/DataFrame API boundary for
**Tasty Bytes Consulting** — a fictional wealth management firm.

- Core date functions: `to_date`, `date_format`, `datediff`, `add_months`, `date_trunc`
- Time-period classification (quarters, fiscal periods, custom rounds)
- Business-day counting via a Python UDF
- `createOrReplaceTempView` — bridging DataFrames to Spark SQL
- `spark.sql()` for complex queries on registered views
- Mixed SQL + DataFrame API workflows applied to price and transaction data

In [ ]:
import sys
import datetime

sys.path.insert(0, '.')
from demo_data import create_spark_session, load_all, COMPANY_NAME

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType

spark = create_spark_session("Date_Utilities_SQL")
spark.sparkContext.setLogLevel("WARN")

data            = load_all(spark)
transactions_df = data["transactions"]
prices_df       = data["prices"]
portfolios_df   = data["portfolios"]
fx_rates_df     = data["fx_rates"]

print(f"Company      : {COMPANY_NAME}")
print(f"Spark        : {spark.version}")
print(f"Transactions : {transactions_df.count()} rows")
print(f"Prices       : {prices_df.count()} rows")

## 1. Core date functions

Parse `txn_date` and `settlement_date` strings to `DateType`, then derive
settlement lag, year-month label, calendar quarter, day-of-week, and
month-start using `datediff`, `date_format`, `date_trunc`, and `add_months`.

In [ ]:
dates_df = (
    transactions_df
    .withColumn("txn_date",            F.to_date(F.col("txn_date"),        "yyyy-MM-dd"))
    .withColumn("settlement_date",     F.to_date(F.col("settlement_date"), "yyyy-MM-dd"))
    .withColumn("settlement_lag_days", F.datediff(F.col("settlement_date"), F.col("txn_date")))
    .withColumn("year_month",          F.date_format(F.col("txn_date"), "yyyy-MM"))
    .withColumn("one_month_later",     F.add_months(F.col("txn_date"), 1))
    .withColumn("quarter",
                F.when(F.month("txn_date") <= 3, "Q1")
                 .when(F.month("txn_date") <= 6, "Q2")
                 .when(F.month("txn_date") <= 9, "Q3")
                 .otherwise("Q4"))
    .withColumn("day_of_week",         F.dayofweek(F.col("txn_date")))
    .withColumn("day_name",            F.date_format(F.col("txn_date"), "EEEE"))
    .withColumn("month_start",         F.date_trunc("month", F.col("txn_date")))
)

print("=== Core date functions on transactions ===")
dates_df.select(
    "txn_id", "txn_date", "settlement_date",
    "settlement_lag_days", "year_month", "one_month_later",
    "quarter", "day_of_week", "day_name", "month_start",
).show(10, truncate=False)
dates_df.printSchema()

## 2. Time-period classification

A closure returning a `DataFrame -> DataFrame` transformer appends calendar
quarters, fiscal halves, and custom business rounds.
Tasty Bytes uses three-round reporting: R1 (Jan-Apr), R2 (May-Aug), R3 (Sep-Dec).

In [ ]:
def add_time_periods(date_col: str = "txn_date"):
    """
    Return a DataFrame -> DataFrame transformer that appends period columns.

    calendar_quarter : Q1-Q4  (Jan-Mar / Apr-Jun / Jul-Sep / Oct-Dec)
    fiscal_half      : H1/H2  (Jan-Jun / Jul-Dec)
    business_round   : R1/R2/R3  (Jan-Apr / May-Aug / Sep-Dec)
    period_label     : e.g. '2024 Q3'
    """
    def _(df: DataFrame) -> DataFrame:
        m = F.month(F.col(date_col))
        y = F.year(F.col(date_col)).cast(StringType())

        cal_q = (
            F.when(m <= 3, "Q1").when(m <= 6, "Q2")
             .when(m <= 9, "Q3").otherwise("Q4")
        )
        fis_h = F.when(m <= 6, "H1").otherwise("H2")
        biz_r = (
            F.when(m <= 4, "R1").when(m <= 8, "R2").otherwise("R3")
        )
        return (
            df
            .withColumn("calendar_quarter", cal_q)
            .withColumn("fiscal_half",      fis_h)
            .withColumn("business_round",   biz_r)
            .withColumn("period_label",     F.concat(y, F.lit(" "), cal_q))
        )
    return _


period_df = dates_df.transform(add_time_periods("txn_date"))

print("=== Time-period classification on transactions ===")
period_df.select(
    "txn_id", "txn_date", "portfolio_id",
    "calendar_quarter", "fiscal_half",
    "business_round",   "period_label",
).show(10, truncate=False)

## 3. Python UDF for business-day counting

Standard equity trades settle T+2. This UDF counts actual weekday business
days between `txn_date` and `settlement_date` so analysts can flag
non-standard settlement lags.

In [ ]:
def count_business_days(start_str: str, end_str: str) -> int:
    """Count weekday business days in the half-open interval [start, end)."""
    if not start_str or not end_str:
        return None
    start   = datetime.datetime.strptime(start_str, "%Y-%m-%d").date()
    end     = datetime.datetime.strptime(end_str,   "%Y-%m-%d").date()
    total, current = 0, start
    while current < end:
        if current.weekday() < 5:
            total += 1
        current += datetime.timedelta(days=1)
    return total


biz_days_udf = F.udf(
    lambda s, e: count_business_days(str(s), str(e)) if s and e else None,
    IntegerType(),
)

settlement_df = dates_df.withColumn(
    "biz_days_to_settle",
    biz_days_udf(
        F.col("txn_date").cast(StringType()),
        F.col("settlement_date").cast(StringType()),
    ),
)

print("=== Settlement lag: calendar days vs business days ===")
settlement_df.select(
    "txn_id", "portfolio_id", "asset_id",
    "txn_date", "settlement_date",
    "settlement_lag_days", "biz_days_to_settle",
).show(15, truncate=False)

## 4. `createOrReplaceTempView` — bridging DataFrames to Spark SQL

Register enriched DataFrames as temp views so SQL queries can reference
all four Tasty Bytes datasets by name.

In [ ]:
enriched_txn_df = (
    transactions_df
    .withColumn("txn_date",            F.to_date(F.col("txn_date"),        "yyyy-MM-dd"))
    .withColumn("settlement_date",     F.to_date(F.col("settlement_date"), "yyyy-MM-dd"))
    .withColumn("settlement_lag_days", F.datediff(F.col("settlement_date"), F.col("txn_date")))
    .withColumn("year_month",          F.date_format(F.col("txn_date"), "yyyy-MM"))
    .withColumn("quarter",
                F.when(F.month("txn_date") <= 3, "Q1")
                 .when(F.month("txn_date") <= 6, "Q2")
                 .when(F.month("txn_date") <= 9, "Q3")
                 .otherwise("Q4"))
)

prices_parsed_df = prices_df.withColumn(
    "price_date", F.to_date(F.col("price_date"), "yyyy-MM-dd")
)

enriched_txn_df.createOrReplaceTempView("vw_transactions")
portfolios_df.createOrReplaceTempView("vw_portfolios")
prices_parsed_df.createOrReplaceTempView("vw_prices")
fx_rates_df.createOrReplaceTempView("vw_fx_rates")

print("Registered views: vw_transactions, vw_portfolios, vw_prices, vw_fx_rates")
spark.sql("SHOW VIEWS").show()

## 5. Spark SQL queries on registered views

Three queries spanning transaction volume, asset-level buy/sell activity,
and side-by-side month-end price performance.

In [ ]:
# Query 1: Monthly transaction volume by portfolio
monthly_vol = spark.sql("""
    SELECT
        t.portfolio_id,
        p.portfolio_name,
        t.year_month,
        COUNT(*)                             AS txn_count,
        ROUND(SUM(t.quantity * t.price), 2)  AS total_notional_usd
    FROM vw_transactions t
    JOIN vw_portfolios   p ON t.portfolio_id = p.portfolio_id
    WHERE t.txn_type IN ('buy', 'sell')
    GROUP BY t.portfolio_id, p.portfolio_name, t.year_month
    ORDER BY t.portfolio_id, t.year_month
""")
print("=== Monthly transaction volume by portfolio ===")
monthly_vol.show(20, truncate=False)
monthly_vol.createOrReplaceTempView("vw_monthly_volume")

# Query 2: YTD buy/sell summary by asset
ytd_summary = spark.sql("""
    SELECT
        asset_id,
        txn_type,
        COUNT(*)                         AS txn_count,
        ROUND(SUM(quantity), 2)          AS total_shares,
        ROUND(SUM(quantity * price), 2)  AS total_notional_usd,
        ROUND(AVG(price), 4)             AS avg_price
    FROM vw_transactions
    WHERE txn_type IN ('buy', 'sell')
    GROUP BY asset_id, txn_type
    ORDER BY asset_id, txn_type
""")
print("=== YTD buy/sell summary by asset ===")
ytd_summary.show(truncate=False)

In [ ]:
# Query 3: Month-end close prices for AAPL (AST-001) and MSFT (AST-004) side by side
price_comparison = spark.sql("""
    SELECT
        aapl.price_date,
        DATE_FORMAT(aapl.price_date, 'yyyy-MM')                                             AS month,
        ROUND(aapl.close_price, 2)                                                          AS aapl_close,
        ROUND(msft.close_price, 2)                                                          AS msft_close,
        ROUND(aapl.close_price - LAG(aapl.close_price) OVER (ORDER BY aapl.price_date), 2) AS aapl_mom_chg,
        ROUND(msft.close_price - LAG(msft.close_price) OVER (ORDER BY aapl.price_date), 2) AS msft_mom_chg
    FROM vw_prices aapl
    JOIN vw_prices msft ON aapl.price_date = msft.price_date
    WHERE aapl.asset_id = 'AST-001'
      AND msft.asset_id = 'AST-004'
    ORDER BY aapl.price_date
""")
print("=== Month-end close prices: AAPL vs MSFT ===")
price_comparison.show(truncate=False)
price_comparison.createOrReplaceTempView("vw_price_comparison")

## 6. Mixed SQL + DataFrame API workflow

A three-step pipeline that fluidly alternates between Spark SQL and the
DataFrame API, registering intermediate results as temp views between steps.

In [ ]:
# Step 1: SQL -- compute monthly close prices with prior-month lag per asset
spark.sql("""
    SELECT
        asset_id,
        price_date,
        DATE_FORMAT(price_date, 'yyyy-MM')                                AS month,
        close_price,
        LAG(close_price) OVER (PARTITION BY asset_id ORDER BY price_date) AS prev_close
    FROM vw_prices
    WHERE asset_id IN ('AST-001', 'AST-002', 'AST-004', 'AST-010')
""").createOrReplaceTempView("vw_monthly_returns_raw")

monthly_ret_df = spark.sql("""
    SELECT
        asset_id,
        month,
        close_price,
        ROUND((close_price - prev_close) / prev_close * 100, 4) AS monthly_return_pct
    FROM vw_monthly_returns_raw
    WHERE prev_close IS NOT NULL
""")

# Step 2: DataFrame API -- rank assets by monthly return using Window functions
ret_window = Window.partitionBy("month").orderBy(F.desc("monthly_return_pct"))

ranked_df = (
    monthly_ret_df
    .withColumn("return_rank",
                F.rank().over(ret_window))
    .withColumn("performance",
                F.when(F.col("return_rank") == 1, "Best")
                 .when(F.col("return_rank") == 2, "Strong")
                 .otherwise("Standard"))
    .orderBy("month", "return_rank")
)

print("=== Monthly asset returns with rankings ===")
ranked_df.show(20, truncate=False)

# Step 3: Back to SQL -- filter and format the final performance report
ranked_df.createOrReplaceTempView("vw_ranked_returns")

final_report = spark.sql("""
    SELECT
        month                                       AS report_month,
        asset_id,
        ROUND(close_price, 2)                       AS close_price_usd,
        CONCAT(ROUND(monthly_return_pct, 2), '%')   AS monthly_return,
        return_rank,
        performance
    FROM vw_ranked_returns
    WHERE performance IN ('Best', 'Strong')
    ORDER BY month, return_rank
""")

print("=== Monthly top performers -- Tasty Bytes Consulting ===")
final_report.show(20, truncate=False)

## 7. Listing and cleaning up temp views

`SHOW VIEWS` lists all active temp views. Drop them when the notebook
finishes to avoid stale state across sessions.

In [ ]:
print("Registered temp views:")
spark.sql("SHOW VIEWS").show()

views_to_drop = [
    "vw_transactions",
    "vw_portfolios",
    "vw_prices",
    "vw_fx_rates",
    "vw_monthly_volume",
    "vw_price_comparison",
    "vw_monthly_returns_raw",
    "vw_ranked_returns",
]
for view in views_to_drop:
    spark.catalog.dropTempView(view)

print("All views dropped.")
spark.sql("SHOW VIEWS").show()

In [ ]:
spark.stop()